# 05 — Retrieval Evaluation

Tier 4 item from `docs/roadmap.md`. Answers two questions with numbers instead of vibes:

1. **Retrieval recall@k** — for questions we already know the answer to, does the retriever actually surface the right source document (and page)?
2. **Routing correctness** — for questions that should *not* be answerable from this knowledge base, does the new conditional edge in `create_sarma_graph` (added for Tier 2) correctly route to `no_context` instead of letting the LLM improvise?

The eval set below isn't invented — the two in-domain cases are the exact questions and correct sources/pages you already validated by hand in `notebooks/04_assistant_test.ipynb`. Turning a manually-checked example into a repeatable eval is the whole point of this notebook.

**Before running:** this assumes the Chroma DB is already built (run `04_assistant_test.ipynb` first if `data/chroma_db` doesn't exist yet).


In [1]:
from sarma.vectorstore.vectorstore import load_vector_store
from sarma.retriever import create_retriever
from sarma.prompts import rag_prompt
from sarma.llm import llm
from sarma.graph.workflow import create_sarma_graph

db = load_vector_store()
retriever = create_retriever(db)
print("Vector store loaded:", db._collection.count(), "chunks")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector store loaded: 3442 chunks


## Eval set

`expected_source` / `expected_pages` are deliberately lenient (a list of acceptable pages, not one exact page) since a document can legitimately answer a question from more than one passage. The two `should_refuse` cases are outside this knowledge base entirely (agronomy + Sentinel-2 remote sensing) and exist to test the Tier 2 routing, not retrieval quality.


In [3]:
EVAL_SET = [
    {
        "question": "What are the recommended nitrogen rates for spring wheat?",
        "expected_source": "RB209 Arable crops.pdf",
        "expected_pages": [32],
        "should_refuse": False,
    },
    {
        "question": "What is Sentinel-2?",
        "expected_source": "Sentinel-2-product-specifications-document-V14-9.pdf",
        "expected_pages": [39, 26, 261, 522, 23],
        "should_refuse": False,
    },
    {
        "question": "What's the weather forecast for London tomorrow?",
        "expected_source": None,
        "expected_pages": [],
        "should_refuse": True,
    },
    {
        "question": "Who won the 2024 football World Cup?",
        "expected_source": None,
        "expected_pages": [],
        "should_refuse": True,
    },
]


## 1. Retrieval recall@k

Source-level recall (did the correct PDF show up at all in the top-k) is the primary number — it's robust and unambiguous. Page-level recall is reported too, but treat it as informative rather than a hard pass/fail: a correct answer can legitimately come from a neighbouring page.


In [4]:
def evaluate_retrieval(db, eval_set, k=5):
    rows = []
    for case in eval_set:
        if case["should_refuse"]:
            continue  # these are for the routing check below, not retrieval recall
        results = db.similarity_search_with_score(case["question"], k=k)
        retrieved_sources = [doc.metadata.get("source") for doc, _ in results]
        retrieved_pages = [doc.metadata.get("page") for doc, _ in results]

        source_hit = case["expected_source"] in retrieved_sources
        page_hit = any(p in case["expected_pages"] for p in retrieved_pages)
        best_score = min(score for _, score in results)

        rows.append({
            "question": case["question"],
            "source_hit": source_hit,
            "page_hit": page_hit,
            "best_score": round(best_score, 4),
        })
    return rows

results = evaluate_retrieval(db, EVAL_SET, k=5)
for r in results:
    print(f"[{'PASS' if r['source_hit'] else 'FAIL'}] source | [{'PASS' if r['page_hit'] else 'FAIL'}] page "
          f"| best_score={r['best_score']:.4f} | {r['question']}")

source_recall = sum(r["source_hit"] for r in results) / len(results)
print(f"\nSource-level recall@5: {source_recall:.0%}")


[PASS] source | [PASS] page | best_score=0.3067 | What are the recommended nitrogen rates for spring wheat?
[PASS] source | [PASS] page | best_score=0.4186 | What is Sentinel-2?

Source-level recall@5: 100%


## 2. Score distribution — use this to set `DEFAULT_SCORE_THRESHOLD`

`nodes.py` currently ships with a placeholder threshold of `0.5`. Run the cell below and look at the gap (or lack of one) between best-scores for in-domain vs. out-of-domain questions — pick a threshold that sits in that gap. If there's no clean gap, that's worth knowing in advance.


In [5]:
for case in EVAL_SET:
    best = db.similarity_search_with_score(case["question"], k=1)[0][1]
    kind = "SHOULD REFUSE" if case["should_refuse"] else "in-domain"
    print(f"{best:.4f}  ({kind:14s})  {case['question']}")


0.3067  (in-domain     )  What are the recommended nitrogen rates for spring wheat?
0.4186  (in-domain     )  What is Sentinel-2?
0.8926  (SHOULD REFUSE )  What's the weather forecast for London tomorrow?
1.0358  (SHOULD REFUSE )  Who won the 2024 football World Cup?


## 3. End-to-end routing check

This exercises the actual `create_sarma_graph` conditional edge added for Tier 2 — not just raw retrieval scores, but whether the compiled graph really routes each question to `generate` or `no_context` as expected.


In [6]:
graph = create_sarma_graph(retriever, rag_prompt, llm)

routing_results = []
for case in EVAL_SET:
    result = graph.invoke({"question": case["question"]})
    routed_to_refusal = result["citations"] == [] and "don't have enough information" in result["answer"].lower()
    correct = routed_to_refusal == case["should_refuse"]
    routing_results.append(correct)
    print(f"[{'PASS' if correct else 'FAIL'}] should_refuse={case['should_refuse']:<5} "
          f"actually_refused={routed_to_refusal:<5} | {case['question']}")
    print(f"   -> {result['answer'][:100]}...")

print(f"\nRouting accuracy: {sum(routing_results)}/{len(routing_results)}")


[PASS] should_refuse=0     actually_refused=0     | What are the recommended nitrogen rates for spring wheat?
   -> The recommended nitrogen rates for spring-sown wheat, based on soil type and SNS Index, are as follo...
[PASS] should_refuse=0     actually_refused=0     | What is Sentinel-2?
   -> Sentinel-2 is an European earth polar-orbiting satellite constellation consisting of two identical s...
[PASS] should_refuse=1     actually_refused=1     | What's the weather forecast for London tomorrow?
   -> I don't have enough information in the knowledge base to answer that question....
[PASS] should_refuse=1     actually_refused=1     | Who won the 2024 football World Cup?
   -> I don't have enough information in the knowledge base to answer that question....

Routing accuracy: 4/4


## Notes

- This is a small, hand-built eval set (n=4) - enough to sanity-check the graph and to have a real number to quote ("recall@5 was X% on a small labelled set"), not a rigorous benchmark. Be ready to say that distinction out loud if asked; overclaiming rigor here is a worse look than being precise about the eval's limits.
- The  next step, if asked "what would you do with more time," is a larger eval set (10-20+ questions per document, covering edge cases like questions the documents *partially* answer) and a faithfulness check (does the generated answer only use retrieved content) alongside retrieval recall - see Part 6.2 of the interview-prep notebook for that pattern.
